# Combined Corpus Train → ISOT Test — DistilBERT 256

Train klasöründeki CSV dosyaları bir kez Drive'a indirilir. Combined Corpus'un %10'u doğrulama için ayrılır; temiz ISOT verisinin tamamından sınıf oranları korunarak rastgele seçilen 10.000 haberle son test yapılır. Metinler her iki veri kümesinde başlıksız kullanılır. GPU seçip hücreleri sırayla çalıştırın.

**Etiket çalışma hipotezi:** Combined Corpus `0 = sahte`, `1 = gerçek`; ISOT `0 = gerçek`, `1 = sahte`. Eğitim kodu kaynak etiketlerini ISOT düzenine çevirir. Bu eşleme doğrudan kaynak tanımıyla henüz doğrulanmadı.


In [ ]:
from pathlib import Path
from google.colab import drive
import subprocess, sys, shutil
drive.mount('/content/drive')
DRIVE_PROJECT = Path('/content/drive/MyDrive/fake_news_detection')
PROJECT = Path('/content/fake-news-detection')
COMBINED_DRIVE = DRIVE_PROJECT / 'data' / 'Combined_Corpus_Train'
# Önceden Drive'a yüklediğiniz hazır ISOT dosyaları:
ISOT_DRIVE = DRIVE_PROJECT / 'data' / 'raw'
OUTPUT_ROOT = DRIVE_PROJECT / 'sonuclar' / 'Combined_Corpus_Train_ISOT_Test'
CHECKPOINT_ROOT = DRIVE_PROJECT / 'checkpointler' / 'Combined_Corpus_Train_ISOT_Test'
print('Combined Train:', COMBINED_DRIVE)
print('ISOT hazır veri:', ISOT_DRIVE)


In [ ]:
if (PROJECT / '.git').is_dir():
    subprocess.run(['git','-C',str(PROJECT),'pull','--ff-only'], check=True)
elif not PROJECT.exists():
    subprocess.run(['git','clone','https://github.com/burak-dursunn/fake-news-detection.git',str(PROJECT)], check=True)
else:
    raise RuntimeError(f'Git deposu olmayan klasör: {PROJECT}')
print('Kod sürümü:', subprocess.check_output(['git','-C',str(PROJECT),'rev-parse','HEAD'], text=True).strip())
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(PROJECT / 'requirements-colab.txt')],check=True)


## Combined Corpus Train dosyalarını indir

Yalnız kaynak deponun Train klasörü indirilir; dosyalar kontrol edilerek Drive'da tutulur. Sonraki çalıştırmalarda tekrar indirilmez. Kaynak Test klasörü bu eğitime katılmaz.


In [ ]:
subprocess.run([sys.executable,'-u','-m','src.download_combined_train','--output',str(COMBINED_DRIVE)], cwd=PROJECT,check=True)


## Hazır ISOT verisini ve GPU'yu kontrol et

ISOT verisinde Reuters ifadesi bulunup bulunmaması seçilen hazır dosyanın sürümüne bağlıdır. Yanlış klasör görünüyorsa ilk hücrede `ISOT_DRIVE` yolunu değiştirin. Verileri değiştirmez veya yeniden temizlemez.


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Çalışma zamanı türünden GPU seçin.')
print('GPU:',torch.cuda.get_device_name(0))
LOCAL_ISOT = Path('/content/isot_combined_test')
LOCAL_ISOT.mkdir(exist_ok=True)
for filename in ['isot_temiz.csv']:
    source = ISOT_DRIVE / filename
    if not source.is_file():
        raise FileNotFoundError(f'ISOT dosyası bulunamadı: {source}. İlk hücrede ISOT_DRIVE yolunu düzeltin.')
    shutil.copy2(source, LOCAL_ISOT / filename)
print('ISOT dosyaları hazır:', LOCAL_ISOT)


## Eğit ve ISOT testini çalıştır

Her çalıştırmada ayrı tarihli klasör açılır. Kayıtlı checkpointler Drive'da kalır; kesinti halinde kod eğitime otomatik devam etmez. Önceki sonuçlar üzerine yazılmaz.


In [ ]:
from datetime import datetime
OUTPUT = OUTPUT_ROOT / ('DistilBERT_256Token_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
config = PROJECT / 'configs' / 'CombinedCorpus_Train_ISOT_Test_DistilBERT_256.json'
command = [sys.executable,'-u','-m','src.train_combined_test_isot',
           '--combined-train',str(COMBINED_DRIVE),'--isot-dir',str(LOCAL_ISOT),
           '--config',str(config),'--output',str(OUTPUT),'--checkpoint-root',str(CHECKPOINT_ROOT)]
print('Sonuç klasörü:', OUTPUT, flush=True)
# Eğitim çıktısını tamamlanmasını beklemeden Colab ekranına aktar.
with subprocess.Popen(command, cwd=PROJECT, stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        return_code = process.wait()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)


In [ ]:
from IPython.display import Image, display
for path in sorted(OUTPUT.glob('*.png')):
    print(path.name)
    display(Image(filename=str(path)))
print('Tüm sonuçlar:', OUTPUT)
